# 🔑 Materi 3 — Teknik Ekstraksi Poin Penting dari Dokumen
### Training NLP

Dokumen kerja sering panjang, padahal yang kita butuhkan hanya **poin pentingnya**. Tiga teknik yang paling sering dipakai di industri:

| Bagian | Teknik | Menjawab Pertanyaan |
|--------|--------|---------------------|
| 3.1–3.2 | **Ekstraksi Kata Kunci** (TF-IDF) | "Dokumen ini membahas apa?" |
| 3.3–3.4 | **NER** (*Named Entity Recognition*) | "Mana tanggal, uang, organisasi, kontaknya?" |
| 3.5–3.6 | **Peringkasan Otomatis** (ekstraktif) | "Apa inti dokumen ini dalam 2–3 kalimat?" |
| 3.7 | **Pipeline gabungan**: laporan singkat otomatis | Semua di atas, sekali jalan |

In [ ]:
# ===== Persiapan =====
!pip install Sastrawi -q

import re
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

daftar_stopword = set(StopWordRemoverFactory().get_stop_words())
print(f"Library siap ✅ ({len(daftar_stopword)} stopword Bahasa Indonesia dimuat)")

## 3.1 Ekstraksi Kata Kunci dengan TF-IDF

Logikanya sederhana: kata dengan **bobot TF-IDF tertinggi** dalam sebuah dokumen = **kata kunci** dokumen tersebut. Stopword kita buang agar kata umum tidak ikut terpilih.

In [ ]:
dokumen_kerja = [
    "penggantian oli mesin dilakukan setiap 4000 km untuk menjaga performa mesin motor",
    "laporan produksi harian mencatat jumlah unit motor yang selesai dirakit di pabrik",
    "pelatihan keselamatan kerja wajib diikuti seluruh karyawan area produksi setiap tahun",
    "sistem booking online mempermudah pelanggan menjadwalkan servis berkala kendaraan",
]

def ekstrak_kata_kunci(daftar_dokumen, jumlah=3):
    """Mengambil N kata kunci terpenting dari setiap dokumen (berbasis TF-IDF)."""
    vec = TfidfVectorizer(stop_words=list(daftar_stopword))
    matriks = vec.fit_transform(daftar_dokumen)
    fitur = vec.get_feature_names_out()

    hasil = []
    for i in range(len(daftar_dokumen)):
        bobot = matriks[i].toarray().flatten()
        urutan_teratas = bobot.argsort()[::-1][:jumlah]   # indeks bobot tertinggi
        hasil.append(", ".join(fitur[j] for j in urutan_teratas))
    return hasil

pd.DataFrame({
    "Dokumen": dokumen_kerja,
    "Kata Kunci (Top 3)": ekstrak_kata_kunci(dokumen_kerja),
})

## 3.2 Melihat "Isi Kepala" TF-IDF — Bobot Setiap Kata

Agar tidak terasa seperti kotak hitam, mari lihat bobot lengkap kata-kata pada **satu dokumen**:

In [ ]:
vec = TfidfVectorizer(stop_words=list(daftar_stopword))
matriks = vec.fit_transform(dokumen_kerja)

# Ambil bobot dokumen pertama, urutkan dari tertinggi
bobot_dok1 = pd.DataFrame({
    "Kata": vec.get_feature_names_out(),
    "Bobot TF-IDF": matriks[0].toarray().flatten().round(3),
})
bobot_dok1 = bobot_dok1[bobot_dok1["Bobot TF-IDF"] > 0].sort_values("Bobot TF-IDF", ascending=False)
print("Dokumen 1:", dokumen_kerja[0])
bobot_dok1.reset_index(drop=True)

💡 Kata *mesin* berbobot tertinggi karena muncul 2× di dokumen ini dan tidak muncul di dokumen lain. **Tidak ada sihir** — hanya perhitungan frekuensi yang cerdas.

## 3.3 NER Sederhana — Mendeteksi Entitas Penting (Rule-Based)

**NER (Named Entity Recognition)** menjawab: *"di dokumen ini, mana yang tanggal? nominal uang? organisasi? kontak?"*

Versi sederhana dibuat dengan pola regex — satu pola per jenis entitas:

In [ ]:
laporan = (
    "Pada 15 Juni 2026, PT Astra Honda Motor mengirim 250 unit sparepart "
    "senilai Rp 1.200.000.000 ke dealer di Surabaya. Konfirmasi penerimaan "
    "dikirim ke gudang.sby@email.com atau 0813-9876-5432 paling lambat 20 Juni 2026, "
    "dengan tingkat kelengkapan minimal 98%."
)

pola_entitas = {
    "TANGGAL":    r"\d{1,2}\s+\w+\s+\d{4}",             # 15 Juni 2026
    "UANG":       r"Rp\s?[\d.]+\d",                       # Rp 1.200.000.000
    "JUMLAH":     r"\d+\s+unit",                          # 250 unit
    "PERSENTASE": r"\d+\s?%",                             # 98%
    "EMAIL":      r"[\w.+-]+@[\w-]+\.[\w.]+",             # gudang.sby@email.com
    "TELEPON":    r"08\d{2}[-\s]?\d{4}[-\s]?\d{4}",       # 0813-9876-5432
    "ORGANISASI": r"PT\s+[A-Z][\w\s]+?Motor",             # PT Astra Honda Motor
}

baris = []
for jenis, pola in pola_entitas.items():
    for hasil in re.findall(pola, laporan):
        baris.append({"Jenis": jenis, "Entitas": hasil})

print("Teks:", laporan, "\n")
pd.DataFrame(baris)

## 3.4 Menandai Entitas Langsung di Dalam Teks

Agar hasil NER mudah diperiksa manusia, kita tandai entitas langsung di teks (seperti stabilo).

Di bagian ini kita pakai dua pendekatan:
1. **Regex (rule-based sederhana):** cepat untuk entitas berpola tetap seperti tanggal, uang, email, dan telepon.
2. **spaCy + EntityRuler (rule-based terstruktur):** konsepnya sama-sama aturan, tetapi manajemen polanya lebih rapi dan mudah dikembangkan.

In [ ]:
def tandai_entitas(teks, pola_entitas):
    """Mengganti setiap entitas yang terdeteksi menjadi [JENIS: entitas]."""
    hasil = teks
    for jenis, pola in pola_entitas.items():
        hasil = re.sub(pola, lambda m: f"[{jenis}: {m.group()}]", hasil)
    return hasil

print(tandai_entitas(laporan, pola_entitas))

In [ ]:
# Versi spaCy: menandai entitas langsung di dalam teks
!pip install spacy -q

import spacy
from spacy.pipeline import EntityRuler

nlp = spacy.blank("id")
ruler = nlp.add_pipe("entity_ruler")

ruler.add_patterns([
    {"label": "TANGGAL", "pattern": [{"TEXT": {"REGEX": "\\d{1,2}"}}, {"LOWER": "juni"}, {"TEXT": {"REGEX": "\\d{4}"}}]},
    {"label": "UANG", "pattern": [{"TEXT": "Rp"}, {"TEXT": {"REGEX": "[0-9.]+"}}]},
    {"label": "JUMLAH", "pattern": [{"TEXT": {"REGEX": "\\d+"}}, {"LOWER": "unit"}]},
    {"label": "PERSENTASE", "pattern": [{"TEXT": {"REGEX": "\\d+"}}, {"TEXT": "%"}]},
    {"label": "EMAIL", "pattern": [{"TEXT": {"REGEX": "[\\w.+-]+@[\\w-]+\\.[\\w.]+"}}]},
    {"label": "TELEPON", "pattern": [{"TEXT": {"REGEX": "08\\d{2}[-\\s]?\\d{4}[-\\s]?\\d{4}"}}]},
    {"label": "ORGANISASI", "pattern": "PT Astra Honda Motor"}
])

def tandai_entitas_spacy(teks):
    doc = nlp(teks)
    hasil = teks
    for ent in reversed(doc.ents):
        hasil = hasil[:ent.start_char] + f"[{ent.label_}: {ent.text}]" + hasil[ent.end_char:]
    return hasil

print(tandai_entitas_spacy(laporan))

### Contoh Level Industri: spaCy dengan Model NER Terlatih

Pada dokumen nyata, entitas seperti **nama orang**, **kota**, atau **organisasi** sering tidak punya pola tetap.
Untuk kasus seperti ini, gunakan model NER terlatih di spaCy agar entitas dapat dikenali dari konteks kalimat, bukan hanya regex.

In [ ]:
# Contoh industri: model NER terlatih spaCy
!pip install spacy -q

import spacy

try:
    nlp_terlatih = spacy.load("xx_ent_wiki_sm")
except OSError:
    !python -m spacy download xx_ent_wiki_sm -q
    nlp_terlatih = spacy.load("xx_ent_wiki_sm")

laporan_industri = (
    "Pada 21 Juli 2026, tim procurement PT Astra Honda Motor menugaskan Budi Santoso "
    "untuk negosiasi kontrak dengan Bosch di Jakarta sebelum presentasi ke manajemen regional Asia."
)

def tandai_entitas_model_terlatih(teks, nlp_model):
    doc = nlp_model(teks)
    hasil = teks
    for ent in reversed(doc.ents):
        hasil = hasil[:ent.start_char] + f"[{ent.label_}: {ent.text}]" + hasil[ent.end_char:]
    return hasil

print(tandai_entitas_model_terlatih(laporan_industri, nlp_terlatih))

💡 **Ringkasnya:** regex dan spaCy `EntityRuler` sama-sama pendekatan berbasis aturan, cocok untuk entitas yang polanya jelas dan konsisten.

💡 **Level industri:** untuk entitas yang tidak berpola tetap (misalnya nama orang, lokasi, produk), biasanya dipakai model NER terlatih di spaCy atau model *transformer* agar akurasi lebih tinggi.

## 3.5 Peringkasan Teks Otomatis (Ekstraktif)

Metode **ekstraktif** = memilih kalimat-kalimat terpenting dari dokumen asli:

1. Hitung **frekuensi kata penting** di seluruh dokumen (tanpa stopword)
2. **Skor setiap kalimat** = jumlah frekuensi kata-kata di dalamnya
3. Ambil kalimat dengan **skor tertinggi** -> itulah ringkasannya

In [ ]:
artikel = (
    "Natural Language Processing membantu perusahaan mengolah dokumen secara otomatis. "
    "Setiap hari perusahaan menerima ratusan dokumen seperti invoice, laporan, dan email. "
    "Membaca dokumen secara manual membutuhkan waktu lama dan rawan kesalahan. "
    "Dengan NLP, komputer dapat mengekstraksi informasi penting dari dokumen dalam hitungan detik. "
    "Teknologi ini meningkatkan efisiensi kerja secara signifikan. "
    "Karyawan pun dapat fokus pada pekerjaan analisis yang bernilai lebih tinggi."
)

# Langkah 1 & 2: hitung frekuensi kata dan skor tiap kalimat
daftar_kalimat = re.split(r"(?<=[.!?])\s+", artikel.strip())
kata_semua = re.findall(r"[a-z]+", artikel.lower())
frek = Counter(k for k in kata_semua if k not in daftar_stopword and len(k) > 3)

def skor_kalimat(kal):
    return sum(frek.get(k, 0) for k in re.findall(r"[a-z]+", kal.lower()))

# Transparansi: lihat skor SEMUA kalimat sebelum memilih
pd.DataFrame({
    "Kalimat": daftar_kalimat,
    "Skor": [skor_kalimat(k) for k in daftar_kalimat],
}).sort_values("Skor", ascending=False).reset_index(drop=True)

In [ ]:
def ringkas(teks, jumlah_kalimat=2):
    """Meringkas teks dengan memilih kalimat berskor tertinggi,
    lalu mengurutkannya kembali sesuai posisi asli agar tetap runtut."""
    kalimat = re.split(r"(?<=[.!?])\s+", teks.strip())
    kata = re.findall(r"[a-z]+", teks.lower())
    frek = Counter(k for k in kata if k not in daftar_stopword and len(k) > 3)

    def skor(kal):
        return sum(frek.get(k, 0) for k in re.findall(r"[a-z]+", kal.lower()))

    terpilih = sorted(kalimat, key=skor, reverse=True)[:jumlah_kalimat]
    return " ".join(k for k in kalimat if k in terpilih)

print("=== ARTIKEL ASLI (6 kalimat) ===")
print(artikel)
print()
print("=== RINGKASAN 2 KALIMAT ===")
print(ringkas(artikel, 2))
print()
print("=== RINGKASAN 3 KALIMAT ===")
print(ringkas(artikel, 3))

## 3.6 Ekstraktif vs Abstraktif

| | **Ekstraktif** (yang kita buat) | **Abstraktif** |
|---|---|---|
| Cara kerja | *Memilih* kalimat asli | *Menulis ulang* dengan kalimat baru |
| Contoh teknologi | TF-IDF, TextRank | LLM (ChatGPT, Claude, Gemini) |
| Kelebihan | Cepat, murah, tidak pernah "mengarang" | Luwes, ringkasannya alami |
| Kekurangan | Kalimatnya kaku (apa adanya) | Butuh model besar, bisa keliru |

## 3.7 Pipeline Gabungan — Laporan Singkat Otomatis

Di dunia kerja, ketiga teknik digabung: satu dokumen masuk → keluar **kata kunci + entitas + ringkasan** sekaligus:

In [ ]:
def laporan_singkat(teks):
    """Pipeline Materi 3: ekstraksi kata kunci + NER + ringkasan dalam sekali jalan."""
    return {
        "Kata Kunci": ekstrak_kata_kunci([teks], jumlah=4)[0],
        "Entitas": [f"{j}: {h}" for j, p in pola_entitas.items() for h in re.findall(p, teks)],
        "Ringkasan": ringkas(teks, 2),
    }

dokumen_masuk = (
    "Pada 15 Juni 2026, PT Astra Honda Motor menyelesaikan pengiriman 250 unit sparepart "
    "senilai Rp 1.200.000.000 ke jaringan dealer di Surabaya. Pengiriman sparepart ini "
    "merupakan bagian dari program penguatan stok dealer menjelang musim servis. "
    "Tim logistik melaporkan tingkat kelengkapan pengiriman mencapai 98%. "
    "Konfirmasi penerimaan dikirim melalui gudang.sby@email.com paling lambat 20 Juni 2026."
)

hasil = laporan_singkat(dokumen_masuk)
print("KATA KUNCI :", hasil["Kata Kunci"])
print()
print("ENTITAS    :")
for e in hasil["Entitas"]:
    print("  -", e)
print()
print("RINGKASAN  :", hasil["Ringkasan"])

✅ **Satu fungsi, tiga wawasan.** Bayangkan pipeline ini berjalan otomatis untuk setiap laporan yang masuk ke email tim Anda.

---
# 🎯 Rangkuman & Latihan Mandiri — Materi 3

| Teknik | Fungsi yang Kita Buat | Kegunaan |
|--------|----------------------|----------|
| Ekstraksi kata kunci | `ekstrak_kata_kunci()` | Menandai topik dokumen otomatis |
| NER rule-based | `pola_entitas` + `tandai_entitas()` | Menarik tanggal, uang, kontak, organisasi |
| Peringkasan ekstraktif | `ringkas()` | Memahami dokumen panjang dalam detik |
| Pipeline gabungan | `laporan_singkat()` | Ketiganya sekali jalan |

### ✍️ Latihan Mandiri
1. Tambahkan jenis entitas baru **"KECEPATAN"** dengan pola untuk teks seperti `4000 km` (petunjuk: `\d+\s?km`).
2. Ganti `artikel` di bagian 3.5 dengan paragraf dari laporan unit kerja Anda, lalu bandingkan ringkasan 2 vs 3 kalimat.
3. Ubah `laporan_singkat()` agar juga mengembalikan **jumlah kalimat** dokumen asli.
4. **Tantangan:** pada tabel skor kalimat (3.5), mengapa kalimat berisi kata *dokumen* cenderung berskor tinggi? Cek `frek` untuk membuktikannya.

---
## 💡 Jawaban Latihan Mandiri

### Latihan 1 — Tambah Entitas "KECEPATAN"

Kita menambahkan pola baru ke `pola_entitas` dan mengujinya pada laporan yang memuat jarak tempuh.

In [ ]:
# Latihan 1: tambah entitas KECEPATAN ke pola_entitas
pola_entitas_baru = pola_entitas.copy()
pola_entitas_baru["KECEPATAN"] = r"\d+\s?km"  # 4000 km, 500 km, 12000 km

laporan_latihan1 = (
    "Pada 15 Juni 2026, PT Astra Honda Motor mengirim 250 unit sparepart "
    "senilai Rp 1.200.000.000 ke dealer di Surabaya. Penggantian oli mesin "
    "dilakukan setiap 4000 km untuk menjaga performa mesin motor."
)

baris = []
for jenis, pola in pola_entitas_baru.items():
    for hasil in re.findall(pola, laporan_latihan1):
        baris.append({"Jenis": jenis, "Entitas": hasil})

print("Teks:", laporan_latihan1, "\n")
pd.DataFrame(baris)

**Penjelasan:** Pola `\d+\s?km` cocok untuk teks seperti `4000 km` atau `500 km`. Angka diikuti spasi opsional lalu `km`. Dengan menambahkan satu baris ke `pola_entitas`, kita bisa mengekstraksi entitas kecepatan dari dokumen otomatis tanpa mengubah kode lainnya.

### Latihan 2 — Ganti Artikel & Bandingkan Ringkasan

Kita mengganti `artikel` dengan teks laporan unit kerja, lalu membandingkan ringkasan 2 vs 3 kalimat.

In [ ]:
# Latihan 2: ganti artikel dengan laporan unit kerja, bandingkan ringkasan
artikel_baru = (
    "Laporan bulanan produksi divisi motor menunjukkan peningkatan volume rakit "
    "sebesar 12 persen dibanding bulan sebelumnya. Tim produksi berhasil merakit "
    "15.000 unit motor dalam periode ini. Peningkatan ini didorong oleh permintaan "
    "tinggi dari jaringan dealer di wilayah Jawa dan Sumatera. Kualitas produk "
    "tetap terjaga dengan tingkat cacat produk di bawah 0.5 persen."
)

print("=== ARTIKEL BARU ===")
print(artikel_baru)
print()

# Bandingkan ringkasan 2 vs 3 kalimat
print("=== RINGKASAN 2 KALIMAT ===")
print(ringkas(artikel_baru, 2))
print()
print("=== RINGKASAN 3 KALIMAT ===")
print(ringkas(artikel_baru, 3))

**Penjelasan:** Dengan artikel yang lebih panjang, perbedaan antara ringkasan 2 dan 3 kalimat jadi lebih terasa. Ringkasan 2 kalimat memberikan inti yang paling padat, sedangkan ringkasan 3 kalimat memberikan konteks tambahan. Di dunia kerja, jumlah kalimat ringkasan disesuaikan dengan kebutuhan — misalnya 2 kalimat untuk email ringkas, 3-4 kalimat untuk briefing manajer.

### Latihan 3 — Ubah `laporan_singkat()` dengan Jumlah Kalimat

Kita memodifikasi pipeline agar mengembalikan jumlah kalimat dokumen asli bersama kata kunci, entitas, dan ringkasan.

In [ ]:
# Latihan 3: tambahkan jumlah kalimat ke laporan_singkat()
def laporan_singkat_lengkap(teks):
    """Pipeline Materi 3: kata kunci + NER + ringkasan + jumlah kalimat."""
    kalimat_list = re.split(r"(?<=[.!?])\s+", teks.strip())
    return {
        "Kata Kunci": ekstrak_kata_kunci([teks], jumlah=4)[0],
        "Entitas": [f"{j}: {h}" for j, p in pola_entitas.items() for h in re.findall(p, teks)],
        "Ringkasan": ringkas(teks, 2),
        "Jumlah Kalimat": len(kalimat_list),
    }

dokumen_uji = (
    "Pada 15 Juni 2026, PT Astra Honda Motor menyelesaikan pengiriman 250 unit sparepart "
    "senilai Rp 1.200.000.000 ke jaringan dealer di Surabaya. Pengiriman sparepart ini "
    "merupakan bagian dari program penguatan stok dealer menjelang musim servis. "
    "Tim logistik melaporkan tingkat kelengkapan pengiriman mencapai 98%. "
    "Konfirmasi penerimaan dikirim melalui gudang.sby@email.com paling lambat 20 Juni 2026."
)

hasil = laporan_singkat_lengkap(dokumen_uji)
print("KATA KUNCI      :", hasil["Kata Kunci"])
print("JUMLAH KALIMAT  :", hasil["Jumlah Kalimat"])
print()
print("ENTITAS         :")
for e in hasil["Entitas"]:
    print("  -", e)
print()
print("RINGKASAN       :", hasil["Ringkasan"])

**Penjelasan:** Dengan menambahkan `Jumlah Kalimat`, pengguna pipeline bisa langsung mengetahui seberapa panjang dokumen asli dan seberapa banyak informasi yang diringkas. Misalnya, jika ringkasan hanya 2 kalimat dari 10 kalimat asli, berarti pipeline berhasil merangkum 80% dokumen menjadi inti yang ringkas.

### Latihan 4 — Analisis: Mengapa Kata "Dokumen" Berskor Tinggi?

Pada tabel skor kalimat di bagian 3.5, kalimat yang mengandung kata *dokumen* cenderung berskor tinggi. Mari kita buktikan dengan mengecek frekuensi kata.

In [ ]:
# Latihan 4: analisis mengapa kata "dokumen" berskor tinggi
print("Frekuensi kata kunci (tanpa stopword, min 4 karakter):")
print("=" * 50)
for kata, jumlah in frek.most_common():
    print(f"  {kata:25s} = {jumlah}")

print()
print("Penjelasan:")
print("Kata 'dokumen' berskor tinggi karena:")
print("1. Muncul di BANYAK kalimat (bukan hanya 1)")
print("2. Panjangnya > 3 karakter sehingga tidak difilter")
print("3. Bukan stopword sehingga bobotnya tinggi")
print()

# Buktikan: hitung di kalimat mana saja kata 'dokumen' muncul
for i, kal in enumerate(daftar_kalimat):
    if "dokumen" in kal.lower():
        print(f"  Kalimat {i}: ...{kal[:60]}...")

**Penjelasan:** Ini terjadi karena metode ekstraktif berbasis frekuensi kata. Kata yang muncul di banyak kalimat (misalnya "dokumen", "perusahaan", "NLP") akan memberikan skor tinggi ke setiap kalimat yang memuatnya. Semakin banyak kata kunci di suatu kalimat, semakin tinggi skornya.

Hal ini menunjukkan kekuatan sekaligus keterbatasan pendekatan ekstraktif: kalimat yang "kaya kata kunci" otomatis terpilih, tetapi kadang kalimat penting yang memakai sinonim justru terlewat. Untuk kebutuhan industri, pendekatan ini bisa dikombinasikan dengan TextRank atau model transformer agar hasilnya lebih akurat.

---
➡️ **Lanjut ke Materi 4:** menemukan dokumen yang tepat dari ribuan dokumen — mesin pencari internal.